# Session 5 — Markov decision processes for an RL-based agent

**DAT 5668 · Module 3, Session 1 · Fundamentals (about 80 minutes)**

**Family this session: RL-based agent.** States, actions, rewards. Use an LLM-based agent only for the part that arrives as language. Session 6 writes both designs for one process.

![Inventory as a Markov decision process](../../assets/mdp-inventory.png)

## Run of show
- 10 min — Recap bandits. Cold call: what did a bandit not model that inventory needs?
- 25 min — States, actions, transitions, rewards, discounting.
- 30 min — Live inventory MDP. One transition, then a one-step value.
- 15 min — When RL is the wrong recommendation. Session 6's memo uses this test.

## Learning goals
- Write an MDP for a repeated operations decision.
- Explain discounting as "later cash is worth less, and later mistakes still count."
- List the data you would need before anyone trains a policy.


## 1. From a bandit to an MDP

A bandit has no memory of stock. Inventory does: ordering today changes tomorrow's state. That is an MDP.

| Piece | Inventory meaning |
| --- | --- |
| State | Units on hand at the start of the day |
| Action | Units ordered this morning, arriving before demand |
| Transition | Next stock = current + order − demand, not below 0, not above capacity |
| Reward | Margin on units sold − holding cost − stockout penalty |
| Discount γ | How much tomorrow's reward counts today (0.95 in the demo) |

RL is appropriate when the decision **repeats**, the reward can be **defined**, and you can survive the **exploration**. A one-time contract review fails this test.


In [ ]:
# Instructional store. Capacity is small so the class can read every state.
CAPACITY = 4
HOLD_COST = 0.2
MARGIN = 2.0
STOCKOUT_PENALTY = 1.5
# Demand 0, 1, or 2, equally likely. In a firm this table is estimated from history.
DEMAND_P = {0: 1 / 3, 1: 1 / 3, 2: 1 / 3}


def next_state(stock, order, demand):
    started = min(CAPACITY, stock + order)
    sales = min(started, demand)
    ending = started - sales
    return ending, sales


def reward(stock, order, demand):
    ending, sales = next_state(stock, order, demand)
    stockout = max(0, demand - min(CAPACITY, stock + order))
    return MARGIN * sales - HOLD_COST * ending - STOCKOUT_PENALTY * stockout


def expected_reward(stock, order):
    """Sum over demand of P(demand) * reward. This is the one-step backup."""
    total = 0.0
    for demand, prob in DEMAND_P.items():
        total += prob * reward(stock, order, demand)
    return total


print("Expected one-step reward from stock=1")
for order in range(3):
    print(f"  order {order}: {expected_reward(1, order):.3f}")


## 2. What "Markov" buys you

Tomorrow's stock depends on today's stock, today's order, and today's demand. It does not depend on last month's path once today's stock is known. If leftover stock spoils based on its age, stock level alone is **not** a Markov state. You would add age to the state.

## 3. Data you would need before training

- A state you can measure before the action (on-hand, not "how the week felt").
- The actions the warehouse can actually perform.
- A reward the CFO agrees with, including the stockout penalty.
- Enough repeated days, or a simulator built from those days.

If any row is missing, Session 6's memo should say **do not automate** or **hybrid** (a person handles the unmeasured part).


In [ ]:
# One simulated week under a base-stock rule: order up to 2, but never above capacity.
def base_stock_order(stock, target=2):
    return max(0, min(2, target - stock))


stock = 1
print("day stock order demand sales end reward")
rng_demands = [1, 2, 0, 2, 1]
for day, demand in enumerate(rng_demands, start=1):
    order = base_stock_order(stock)
    ending, sales = next_state(stock, order, demand)
    pay = reward(stock, order, demand)
    print(f"{day:3} {stock:5} {order:5} {demand:6} {sales:5} {ending:3} {pay:7.2f}")
    stock = ending


## Bridge to Session 6

Students map **one** process twice: as an LLM-agent design (tools, knowledge, handoffs) and as an MDP (states, actions, rewards). The deliverable is a go/no-go memo: LLM, RL, hybrid, or do not automate.

The process they will use, unless you assign another, is **replenishment exceptions**: the daily order is numeric, but a promo email or a supplier delay is a document a person or an LLM agent must read.


## Fill this with me

Stock is empty, you order nothing, and two customers show up. Guess the reward, then run it.


In [ ]:
# FILL WITH THE CLASS. A number.
expected_reward_guess = None
actual = reward(0, 0, 2)
print("You expected", expected_reward_guess, "| reward() returned", round(actual, 2))
print("The stockout penalty is the business talking. Change STOCKOUT_PENALTY above and rerun this cell.")
